# Predicting Student Test Scores
Playground Series - Season 6 Episode 1

Predicting student exam performance is helful for understanding how academic habits, lifestyle choices, and learning environments shape outcomes. By analyzing factors such as study hours, class attendance, sleep quality, internet access, and study methods, educators can identify which variables most strongly influence success. Insights from such predictive analysis help institutions design targeted interventions to enhance learning support and improve overall academic achievement.

Yao Yan, Walter Reade, Elizabeth Park. Predicting Student Test Scores. https://kaggle.com/competitions/playground-series-s6e1, 2025. Kaggle.

In [1]:
# imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import PolynomialFeatures
from itertools import combinations
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
import optuna

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

## About the data

| **Column Name**     | **Description**                                                                 |
|----------------------|---------------------------------------------------------------------------------|
| `id`                | A unique identifier assigned to each student record.                            |
| `age`               | The age of the student, represented in years.                                   |
| `gender`            | The gender of the student (e.g., male, female, non-binary, unspecified).        |
| `course`            | The course or academic program the student is enrolled in.                      |
| `study_hours`       | Average number of hours the student studies per day.                            |
| `class_attendance`  | Percentage of attended classes over the total number of scheduled classes.       |
| `internet_access`   | Indicates whether the student has reliable internet access (e.g., yes/no or scale). |
| `sleep_hours`       | Average number of hours of sleep the student gets per night.                    |
| `sleep_quality`     | Perceived quality of sleep, often measured on a numerical or categorical scale.  |
| `study_method`      | The primary study approach used by the student (e.g., solo, group, mixed).       |
| `facility_rating`   | Rating of the institution’s facilities as perceived by the student.              |
| `exam_difficulty`   | The perceived or assigned difficulty level of the exam attempted.                |
| `exam_score`        | The final exam score achieved by the student, ranging from 0 to 100.            |


In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s6e1/sample_submission.csv')

In [4]:
def downcasting(data: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    mem_before = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {mem_before:.2f} MB")
    
    for col in data.select_dtypes(include=["number"]).columns:
        if pd.api.types.is_integer_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="integer")
        elif pd.api.types.is_float_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="float")
    
    mem_after = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {mem_after:.2f} MB")
        print(f"Decreased by {(100 * (mem_before - mem_after) / mem_before):.1f}%\n")
    
    return data

print("Train train:")
train = downcasting(train)
print("Test train:")
test = downcasting(test)

Train train:
Memory usage of dataframe is 62.48 MB
Memory usage after optimization is: 46.26 MB
Decreased by 26.0%

Test train:
Memory usage of dataframe is 24.72 MB
Memory usage after optimization is: 18.80 MB
Decreased by 24.0%



In [5]:
target = "exam_score"
cols = train.drop(columns=["id", target]).columns.tolist()

# Categorical features
cat_cols = [c for c in cols if train[c].dtype in ["object","category"]]
# Numerical features
num_cols = [c for c in cols if train[c].dtype not in ["object","category","bool"]]

print("Categorical cols:")
print(cat_cols)
print("Numerical cols:")
print(num_cols)
print("\n")
for col in cat_cols:
    unique_values = train[col].dropna().unique()  # or orig[col] depending on your dataset
    print(f"Unique values in {col}: {unique_values}")

Categorical cols:
['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
Numerical cols:
['age', 'study_hours', 'class_attendance', 'sleep_hours']


Unique values in gender: ['female' 'other' 'male']
Unique values in course: ['b.sc' 'diploma' 'bca' 'b.com' 'ba' 'bba' 'b.tech']
Unique values in internet_access: ['no' 'yes']
Unique values in sleep_quality: ['average' 'poor' 'good']
Unique values in study_method: ['online videos' 'self-study' 'coaching' 'group study' 'mixed']
Unique values in facility_rating: ['low' 'medium' 'high']
Unique values in exam_difficulty: ['easy' 'moderate' 'hard']


## Feature Engineering

**Feature Engineering Treatments:**

- **gender**: One-hot encoding (nominal: female/other/male)
- **course**: One-hot encoding (nominal: 7 categories - b.sc, diploma, etc.)
- **internet_access**: Binary encoding (yes/no → 1/0)
- **sleep_quality**: Ordinal encoding (poor < average < good)
- **study_method**: One-hot encoding (nominal: 5 methods)
- **facility_rating**: Ordinal encoding (low < medium < high)
- **exam_difficulty**: Ordinal encoding (easy < moderate < hard)


In [6]:
from sklearn.preprocessing import PolynomialFeatures
from itertools import combinations

# Numerical feature engineering
def create_pairwise_interactions(df, numerical_cols):
    """Pairwise numerical interactions (A * B)."""
    df_new = df.copy()
    for col1, col2 in combinations(numerical_cols, 2):
        df_new[f"{col1}_x_{col2}"] = df_new[col1] * df_new[col2]
    return df_new

def create_polynomial_features(df, numerical_cols, degree=2):
    """Polynomial features including squares."""
    df_new = df.copy()
    poly = PolynomialFeatures(degree=degree, interaction_only=False, include_bias=False)
    poly_features = poly.fit_transform(df_new[numerical_cols])
    df_new[poly.get_feature_names_out(numerical_cols)] = poly_features
    return df_new

def create_threeway_interactions(df, numerical_cols):
    """Three-way interactions (A * B * C)."""
    df_new = df.copy()
    for col1, col2, col3 in combinations(numerical_cols, 3):
        df_new[f"{col1}_x_{col2}_x_{col3}"] = df_new[col1] * df_new[col2] * df_new[col3]
    return df_new

def create_ratio_features(df, numerical_cols):
    """Ratio features (A / B)."""
    df_new = df.copy()
    for col1, col2 in combinations(numerical_cols, 2):
        df_new[f"{col1}_div_{col2}"] = df_new[col1] / (df_new[col2] + 1e-8)
    return df_new

In [7]:
# Categorical feature engineering
ORDINAL_MAPPINGS = {
    "sleep_quality": {"poor": 0, "average": 1, "good": 2},
    "facility_rating": {"low": 0, "medium": 1, "high": 2},
    "exam_difficulty": {"easy": 0, "moderate": 1, "hard": 2}
}

def ordinal_encode(df):
    """Apply predefined ordinal mappings."""
    df_new = df.copy()
    for col, mapping in ORDINAL_MAPPINGS.items():
        df_new[f"{col}_num"] = df_new[col].map(mapping).fillna(1)
    return df_new

def to_category(df):
    """Convert object columns to category dtype."""
    df_new = df.copy()
    for c in df_new.select_dtypes(include=['object']).columns:
        df_new[c] = df_new[c].astype("category")
    return df_new

In [8]:
# Parent function
def preprocess_features(df, numerical_cols, 
                       pairwise=False, poly=False, threeway=False, ratios=False,
                       ordinal=True, categorical=True):
    """Complete feature engineering pipeline."""
    df_processed = df.copy()

    # Numerical interactions
    if pairwise:
        df_processed = create_pairwise_interactions(df_processed, numerical_cols)
    if poly:
        df_processed = create_polynomial_features(df_processed, numerical_cols)
    if threeway:
        df_processed = create_threeway_interactions(df_processed, numerical_cols)
    if ratios:
        df_processed = create_ratio_features(df_processed, numerical_cols)
    
    # Categorical preprocessing
    if ordinal:
        df_processed = ordinal_encode(df_processed)
    
    if categorical:
        df_processed = to_category(df_processed)

    return df_processed

## Modelling

In [9]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["id", target])
y = train[target]

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert back to DataFrames with original column names
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df = pd.DataFrame(X_test, columns=X.columns)

# Now apply feature engineering
X_train = preprocess_features(X_train_df, num_cols).reset_index(drop=True)
X_test = preprocess_features(X_test_df, num_cols).reset_index(drop=True)

In [10]:
%%skip
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns
cat_features_idx = [X_train.columns.get_loc(c) for c in cat_cols]

# Simple CatBoost model
model = CatBoostRegressor(
    loss_function="RMSE",
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=0,
    cat_features=cat_features_idx
)

model.fit(X_train, y_train)

# Evaluate on hold-out test set
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")

In [11]:
%%skip
# Hyperparameter Tuning
import optuna

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns
cat_features_idx = [X_train.columns.get_loc(c) for c in cat_cols]

def objective(trial):
    params = {
        'loss_function': 'RMSE', 
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'verbose': False, 
        'cat_features': cat_features_idx,
        'early_stopping_rounds': 50 
    }
    
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)
    
    preds = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, preds))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)


best_params = {
    **study.best_params,
    'cat_features': cat_features_idx,
    'verbose': 0
}
final_model = CatBoostRegressor(**best_params)
final_model.fit(X_train, y_train)

## Prediction

In [12]:
X_final = preprocess_features(X, num_cols).reset_index(drop=True)
test = test.drop(columns=["id"])
test_final = preprocess_features(test, num_cols).reset_index(drop=True)

In [17]:
X_final.columns

Index(['age', 'gender', 'course', 'study_hours', 'class_attendance',
       'internet_access', 'sleep_hours', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty', 'sleep_quality_num',
       'facility_rating_num', 'exam_difficulty_num'],
      dtype='object')

In [16]:
test_final.columns

Index(['age', 'gender', 'course', 'study_hours', 'class_attendance',
       'internet_access', 'sleep_hours', 'sleep_quality', 'study_method',
       'facility_rating', 'exam_difficulty', 'sleep_quality_num',
       'facility_rating_num', 'exam_difficulty_num'],
      dtype='object')

In [13]:
cat_cols = X_final.select_dtypes(include=["object", "category"]).columns
cat_features_idx = [X_train.columns.get_loc(c) for c in cat_cols]

params = {
    'iterations': 1990,
    'learning_rate': 0.285,
    'depth': 5,
    'l2_leaf_reg': 8.427,
    'random_strength': 8.797,
    'bagging_temperature': 0.412,
    'verbose': False,
    'cat_features': cat_features_idx
}

# Initialize model
model = CatBoostRegressor(**params)

model.fit(X_final, y)
predictions = model.predict(test_final)

CatBoostError: catboost/libs/data/model_dataset_compatibility.cpp:72: Feature sleep_quality_num is present in model but not in pool.

In [19]:
submission['exam_score'] = predictions
submission.to_csv('submission.csv', index=False)
submission.head()

,id,exam_score
0,630000,72.092370
1,630001,71.120878
2,630002,89.656523
3,630003,56.878860
4,630004,47.264693
